# Corrected Weighted Fusion — 4 Model Ailesi

Bu notebook dört model ailesinde de aynı validation tabanlı ağırlık formülünü kullanır:

\[
r_i = \max(AUC_{val,i}-0.5, \epsilon)
\]

\[
w_i = rac{r_i}{\sum_j r_j}
\]

Modeller:
- Xception
- EfficientNet-B0
- Swin V2 Tiny
- Swin V2 Tiny + Texture

**Not:** Modeller yeniden eğitilmez. Mevcut ortak test kohortu tahminleri üzerinden weighted score-level fusion yeniden hesaplanır.


In [ ]:
# -*- coding: utf-8 -*-
"""
CORRECTED validation-weighted late fusion — 4 model family

Models:
- Xception
- EfficientNet-B0
- Swin V2 Tiny
- Swin V2 Tiny + Texture

Amaç:
Dört model ailesinin tamamında AYNI ağırlık formülünü kullanmak:

    r_i = max(AUC_val_i - 0.5, epsilon)
    w_i = r_i / sum_j(r_j)

Bu script modelleri yeniden eğitmez.
Mevcut ortak-kohort test olasılıklarını kullanarak weighted score-level fusion'ı
yeniden hesaplar.

ÖNEMLİ:
- Eski hatalı fusion_probability sütunları ASLA kullanılmaz.
- Sadece p_eye / p_brow / p_mouth + label kullanılır.
- Threshold = 0.50 sabit tutulur.
- Ortak test kohortu beklenen n = 196'dır.
- Bootstrap %95 CI paired olarak mouth-only vs fusion karşılaştırması için hesaplanır.
"""

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

In [ ]:
# ============================================================
# 1) GOOGLE DRIVE

In [ ]:
# ============================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Colab dışında çalışıyorsan Drive mount adımını atlayabilirsin.")

In [ ]:
# ============================================================
# 2) ESKİ 3-MODEL WEIGHTED RUN

In [ ]:
# ============================================================
OLD_RUN = Path(
    "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/"
    "Sonuçlar/Fusion_Experiments/02_weighted_soft_voting/"
    "20260809_163157_934154_weighted_soft_voting_seed42"
)

OUT_ROOT = OLD_RUN.parent / "04_weighted_soft_voting_CORRECTED_ALL4_seed42"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# 3) DOĞRU VALIDATION ROC-AUC DEĞERLERİ

In [ ]:
# ============================================================
# Xception değerleri kendi bölgesel deney/fusion raporundaki validation ROC-AUC'lerdir.
# Diğer üç modelde eski hatalı weight-source eşlemesi kullanılmıyor.
VAL_AUC = {
    "xception": {
        "eye":   0.7252,
        "brow":  0.6710,
        "mouth": 0.8228,
    },
    "efficientnet_b0": {
        "eye":   0.7129261038663921,
        "brow":  0.6819407008086252,
        "mouth": 0.7968885838480896,
    },
    "swinv2_tiny": {
        "eye":   0.8453900709219859,
        "brow":  0.7190989603388525,
        "mouth": 0.8858384808968199,
    },
    "swinv2_texture": {
        "eye":   0.7222374742621827,
        "brow":  0.7279553330766270,
        "mouth": 0.8603752001830244,
    },
}

In [ ]:
# ============================================================
# 4) TEST PREDICTION DOSYALARI

In [ ]:
# ============================================================
TEST_FILES = {
    "efficientnet_b0":
        OLD_RUN / "efficientnet_b0" / "predictions" / "aligned_test_predictions.csv",

    "swinv2_tiny":
        OLD_RUN / "swinv2_tiny" / "predictions" / "aligned_test_predictions.csv",

    "swinv2_texture":
        OLD_RUN / "swinv2_texture" / "predictions" / "aligned_test_predictions.csv",
}

# Xception'ın eski fusion çıktısı önce aşağıdaki yerlerde aranır.
# Bulunamazsa Colab dosya yükleme penceresi açılır.
XCEPTION_CANDIDATES = [
    Path("/content/Xception_Fusion_Results/xception_weighted_fusion_test_predictions.csv"),
    Path("/content/xception_weighted_fusion_test_predictions.csv"),
    Path(
        "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/"
        "Sonuçlar/Fusion_Experiments/Xception_Fusion_Results/"
        "xception_weighted_fusion_test_predictions.csv"
    ),
]

EPS = 1e-12
THRESHOLD = 0.50
BOOTSTRAP_ITER = 5000
SEED = 42
EXPECTED_N = 196

In [ ]:
# ============================================================
# 5) YARDIMCI FONKSİYONLAR

In [ ]:
# ============================================================
def compute_weights(aucs):
    """
    TÜM DÖRT MODELDE AYNI FORMÜL:
      r_i = max(AUC_i - 0.5, epsilon)
      w_i = r_i / sum(r)
    """
    raw = {
        region: max(float(auc) - 0.5, EPS)
        for region, auc in aucs.items()
    }

    denom = sum(raw.values())
    if denom <= 0:
        raise RuntimeError("Ağırlık toplamı 0 veya negatif olamaz.")

    weights = {k: v / denom for k, v in raw.items()}

    if not np.isclose(sum(weights.values()), 1.0, atol=1e-10):
        raise RuntimeError("Normalize ağırlıkların toplamı 1 değil.")

    return raw, weights


def _normalized_colname(name):
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
    )


def _find_column(df, candidates):
    lookup = {_normalized_colname(c): c for c in df.columns}

    for cand in candidates:
        key = _normalized_colname(cand)
        if key in lookup:
            return lookup[key]

    return None


def standardize_prediction_table(df, family):
    """
    Üç-model aligned CSV formatını ve eski Xception fusion CSV formatındaki
    muhtemel kolon adlarını ortak şemaya dönüştürür.

    Çıktı:
      fusion_key, p_eye, p_brow, p_mouth, label
    """

    label_col = _find_column(
        df,
        [
            "label", "y_true", "true_label", "target",
            "ground_truth", "ground_truth_label"
        ],
    )

    eye_col = _find_column(
        df,
        [
            "p_eye", "p_fake_eye", "eye_probability", "eye_prob",
            "prob_eye", "probability_eye", "eye_p_fake"
        ],
    )

    brow_col = _find_column(
        df,
        [
            "p_brow", "p_eyebrow", "p_fake_brow", "p_fake_eyebrow",
            "brow_probability", "eyebrow_probability",
            "brow_prob", "eyebrow_prob", "prob_brow", "prob_eyebrow",
            "brow_p_fake", "eyebrow_p_fake"
        ],
    )

    mouth_col = _find_column(
        df,
        [
            "p_mouth", "p_fake_mouth", "mouth_probability", "mouth_prob",
            "prob_mouth", "probability_mouth", "mouth_p_fake"
        ],
    )

    key_col = _find_column(
        df,
        ["fusion_key", "sample_key", "key", "sample_id"]
    )

    missing = []
    if label_col is None:
        missing.append("label / y_true")
    if eye_col is None:
        missing.append("p_eye")
    if brow_col is None:
        missing.append("p_brow / p_eyebrow")
    if mouth_col is None:
        missing.append("p_mouth")

    if missing:
        raise RuntimeError(
            f"\n{family}: gerekli kolonlar bulunamadı: {missing}\n"
            f"Dosyadaki kolonlar:\n{list(df.columns)}\n\n"
            "Xception için eski final prediction CSV'nin üç ayrı bölgesel "
            "olasılığı içerdiğinden emin ol."
        )

    out = pd.DataFrame({
        "p_eye": pd.to_numeric(df[eye_col], errors="raise"),
        "p_brow": pd.to_numeric(df[brow_col], errors="raise"),
        "p_mouth": pd.to_numeric(df[mouth_col], errors="raise"),
        "label": pd.to_numeric(df[label_col], errors="raise").astype(int),
    })

    if key_col is not None:
        out.insert(0, "fusion_key", df[key_col].astype(str).values)
    else:
        # Eski Xception final CSV zaten 196 ortak/eşleşmiş satır ise,
        # anahtar metrik hesabı için zorunlu değildir.
        out.insert(
            0,
            "fusion_key",
            [f"{family}_{i:03d}" for i in range(len(out))]
        )

    return out


def locate_or_upload_xception_csv():
    for path in XCEPTION_CANDIDATES:
        if path.exists():
            print(f"Xception CSV bulundu: {path}")
            return path

    print(
        "\nXception'ın eski prediction CSV'si otomatik bulunamadı.\n"
        "Lütfen şu dosyayı yükle:\n"
        "  xception_weighted_fusion_test_predictions.csv\n"
    )

    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as exc:
        raise FileNotFoundError(
            "Xception CSV bulunamadı ve otomatik Colab upload açılamadı.\n"
            "XCEPTION_CANDIDATES listesine dosyanın gerçek yolunu ekle."
        ) from exc

    if not uploaded:
        raise FileNotFoundError("Xception CSV yüklenmedi.")

    # Tercihen doğru isimli dosyayı seç.
    preferred = "xception_weighted_fusion_test_predictions.csv"
    if preferred in uploaded:
        return Path("/content") / preferred

    # Tek dosya yüklenmişse onu kullan.
    first = next(iter(uploaded.keys()))
    return Path("/content") / first


def calculate_metrics(y_true, prob, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    y_pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "n": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, prob)),
        "pr_auc": float(average_precision_score(y_true, prob)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def bootstrap_auc_ci_and_delta(
    y_true,
    mouth_prob,
    fusion_prob,
    n_boot=5000,
    seed=42,
):
    """
    Paired bootstrap:
    - mouth-only ROC-AUC %95 CI
    - weighted fusion ROC-AUC %95 CI
    - Delta AUC = fusion - mouth %95 CI
    """

    y_true = np.asarray(y_true).astype(int)
    mouth_prob = np.asarray(mouth_prob, dtype=float)
    fusion_prob = np.asarray(fusion_prob, dtype=float)

    rng = np.random.default_rng(seed)
    n = len(y_true)

    mouth_aucs = []
    fusion_aucs = []
    deltas = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        y_b = y_true[idx]

        # Bootstrap örneğinde tek sınıf oluşursa AUC tanımsızdır.
        if np.unique(y_b).size < 2:
            continue

        m_auc = roc_auc_score(y_b, mouth_prob[idx])
        f_auc = roc_auc_score(y_b, fusion_prob[idx])

        mouth_aucs.append(m_auc)
        fusion_aucs.append(f_auc)
        deltas.append(f_auc - m_auc)

    if not deltas:
        raise RuntimeError("Bootstrap için geçerli örnek üretilemedi.")

    def ci95(values):
        lo, hi = np.percentile(values, [2.5, 97.5])
        return float(lo), float(hi)

    mouth_ci = ci95(mouth_aucs)
    fusion_ci = ci95(fusion_aucs)
    delta_ci = ci95(deltas)

    return {
        "bootstrap_iterations_valid": int(len(deltas)),
        "mouth_auc_ci95_low": mouth_ci[0],
        "mouth_auc_ci95_high": mouth_ci[1],
        "fusion_auc_ci95_low": fusion_ci[0],
        "fusion_auc_ci95_high": fusion_ci[1],
        "delta_auc_ci95_low": delta_ci[0],
        "delta_auc_ci95_high": delta_ci[1],
        "delta_ci_excludes_zero": bool(
            delta_ci[0] > 0 or delta_ci[1] < 0
        ),
    }


def validate_common_cohort(df, family):
    if df["fusion_key"].duplicated().any():
        raise RuntimeError(f"{family}: fusion_key tekrarları bulundu.")

    labels = set(df["label"].dropna().unique())
    if not labels.issubset({0, 1}):
        raise RuntimeError(
            f"{family}: label yalnızca 0/1 olmalı. Bulunan: {sorted(labels)}"
        )

    for c in ["p_eye", "p_brow", "p_mouth"]:
        if df[c].isna().any():
            raise RuntimeError(f"{family}: {c} içinde NaN var.")
        if not df[c].between(0, 1).all():
            raise RuntimeError(f"{family}: {c} [0,1] dışında değer içeriyor.")

    if len(df) != EXPECTED_N:
        raise RuntimeError(
            f"{family}: ortak kohort n={len(df)}; beklenen n={EXPECTED_N}."
        )

    n_real = int((df["label"] == 0).sum())
    n_fake = int((df["label"] == 1).sum())

    if (n_real, n_fake) != (101, 95):
        print(
            f"UYARI {family}: sınıf dağılımı REAL={n_real}, FAKE={n_fake}; "
            "beklenen REAL=101, FAKE=95."
        )

In [ ]:
# ============================================================
# 6) XCEPTION DOSYASINI EKLE

In [ ]:
# ============================================================
TEST_FILES["xception"] = locate_or_upload_xception_csv()

# Çalışma sırası makaledeki model sırasına yakın olsun.
MODEL_ORDER = [
    "xception",
    "efficientnet_b0",
    "swinv2_tiny",
    "swinv2_texture",
]

In [ ]:
# ============================================================
# 7) ANA ÇALIŞMA

In [ ]:
# ============================================================
all_metric_rows = []
all_weight_rows = []

for family in MODEL_ORDER:

    print("\n" + "=" * 80)
    print("MODEL:", family)
    print("=" * 80)

    test_path = TEST_FILES[family]

    if not test_path.exists():
        raise FileNotFoundError(
            f"\nDosya bulunamadı:\n{test_path}\n"
        )

    raw_df = pd.read_csv(test_path)
    df = standardize_prediction_table(raw_df, family)
    validate_common_cohort(df, family)

    raw_scores, weights = compute_weights(VAL_AUC[family])

    print("\nValidation ROC-AUC:")
    print(f"  eye   = {VAL_AUC[family]['eye']:.9f}")
    print(f"  brow  = {VAL_AUC[family]['brow']:.9f}")
    print(f"  mouth = {VAL_AUC[family]['mouth']:.9f}")

    print("\nORTAK FORMÜL ile ağırlıklar:")
    print(f"  eye   = {weights['eye']:.6f}")
    print(f"  brow  = {weights['brow']:.6f}")
    print(f"  mouth = {weights['mouth']:.6f}")
    print(f"  sum   = {sum(weights.values()):.6f}")

    # Eski fusion_probability KULLANILMIYOR.
    df["fusion_probability_corrected"] = (
        weights["eye"] * df["p_eye"]
        + weights["brow"] * df["p_brow"]
        + weights["mouth"] * df["p_mouth"]
    )

    fusion_metrics = calculate_metrics(
        df["label"],
        df["fusion_probability_corrected"],
        threshold=THRESHOLD,
    )

    mouth_metrics = calculate_metrics(
        df["label"],
        df["p_mouth"],
        threshold=THRESHOLD,
    )

    delta_auc = (
        fusion_metrics["roc_auc"]
        - mouth_metrics["roc_auc"]
    )

    bootstrap = bootstrap_auc_ci_and_delta(
        df["label"].to_numpy(),
        df["p_mouth"].to_numpy(),
        df["fusion_probability_corrected"].to_numpy(),
        n_boot=BOOTSTRAP_ITER,
        seed=SEED,
    )

    print("\nCorrected weighted fusion:")
    print(f"  n             : {fusion_metrics['n']}")
    print(f"  Accuracy      : {fusion_metrics['accuracy']:.4f}")
    print(f"  Balanced Acc  : {fusion_metrics['balanced_accuracy']:.4f}")
    print(f"  Precision     : {fusion_metrics['precision']:.4f}")
    print(f"  Recall        : {fusion_metrics['recall']:.4f}")
    print(f"  Specificity   : {fusion_metrics['specificity']:.4f}")
    print(f"  F1            : {fusion_metrics['f1']:.4f}")
    print(f"  ROC-AUC       : {fusion_metrics['roc_auc']:.4f}")
    print(f"  PR-AUC        : {fusion_metrics['pr_auc']:.4f}")

    print("\nSame-cohort mouth-only:")
    print(f"  ROC-AUC       : {mouth_metrics['roc_auc']:.4f}")

    print("\nDelta AUC (fusion - mouth):")
    print(f"  point estimate: {delta_auc:+.4f}")
    print(
        "  paired bootstrap %95 CI: "
        f"[{bootstrap['delta_auc_ci95_low']:+.4f}, "
        f"{bootstrap['delta_auc_ci95_high']:+.4f}]"
    )

    family_out = OUT_ROOT / family
    family_out.mkdir(parents=True, exist_ok=True)

    # Tahminleri kaydet.
    df.to_csv(
        family_out / "corrected_weighted_fusion_predictions.csv",
        index=False,
    )

    weight_row = {
        "model_family": family,
        "weight_formula":
            "max(validation_roc_auc - 0.5, epsilon), normalized_to_sum_1",
        "validation_auc_eye": VAL_AUC[family]["eye"],
        "validation_auc_brow": VAL_AUC[family]["brow"],
        "validation_auc_mouth": VAL_AUC[family]["mouth"],
        "raw_eye": raw_scores["eye"],
        "raw_brow": raw_scores["brow"],
        "raw_mouth": raw_scores["mouth"],
        "weight_eye": weights["eye"],
        "weight_brow": weights["brow"],
        "weight_mouth": weights["mouth"],
    }
    all_weight_rows.append(weight_row)

    metric_row = {
        "model_family": family,
        "method": "corrected_validation_weighted_soft_voting_common_formula",
        **fusion_metrics,
        "mouth_only_roc_auc_same_cohort": mouth_metrics["roc_auc"],
        "delta_auc_fusion_minus_mouth": delta_auc,
        **bootstrap,
    }
    all_metric_rows.append(metric_row)

    with open(
        family_out / "corrected_weighted_run_summary.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "model_family": family,
                "method":
                    "corrected_validation_weighted_soft_voting_common_formula",
                "n_common_cohort": EXPECTED_N,
                "decision_threshold": THRESHOLD,
                "weight_formula":
                    "max(validation_roc_auc - 0.5, epsilon), normalized to sum=1",
                "validation_auc": VAL_AUC[family],
                "weights": weights,
                "fusion_metrics": fusion_metrics,
                "mouth_only_metrics_same_cohort": mouth_metrics,
                "delta_auc_fusion_minus_mouth": delta_auc,
                "paired_bootstrap": bootstrap,
                "source_test_predictions": str(test_path),
                "important_note":
                    "Old fusion_probability was ignored. "
                    "Only regional probabilities and labels were reused.",
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

In [ ]:
# ============================================================
# 8) TOPLU SONUÇ TABLOLARI

In [ ]:
# ============================================================
weights_df = pd.DataFrame(all_weight_rows)
metrics_df = pd.DataFrame(all_metric_rows)

weights_df.to_csv(
    OUT_ROOT / "corrected_validation_weights_ALL4.csv",
    index=False,
)

metrics_df.to_csv(
    OUT_ROOT / "corrected_weighted_fusion_metrics_ALL4.csv",
    index=False,
)

# Makaleye kolay kopyalanabilecek özet tablo
paper_df = metrics_df[
    [
        "model_family",
        "n",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "roc_auc",
        "pr_auc",
        "mouth_only_roc_auc_same_cohort",
        "delta_auc_fusion_minus_mouth",
        "delta_auc_ci95_low",
        "delta_auc_ci95_high",
        "delta_ci_excludes_zero",
    ]
].copy()

paper_df.to_csv(
    OUT_ROOT / "paper_ready_corrected_fusion_table.csv",
    index=False,
)

print("\n\n" + "=" * 80)
print("DÖRT MODEL TAMAMLANDI")
print("=" * 80)

print("\nAĞIRLIKLAR:")
print(
    weights_df[
        ["model_family", "weight_eye", "weight_brow", "weight_mouth"]
    ].to_string(index=False)
)

print("\nMAKALE İÇİN ANA SONUÇLAR:")
print(
    paper_df[
        [
            "model_family",
            "n",
            "roc_auc",
            "mouth_only_roc_auc_same_cohort",
            "delta_auc_fusion_minus_mouth",
            "delta_auc_ci95_low",
            "delta_auc_ci95_high",
        ]
    ].to_string(index=False)
)

print("\nÇıktı klasörü:")
print(OUT_ROOT)

print(
    "\nNOT: Makaledeki weighted fusion tablosunda bundan sonra "
    "yalnız bu corrected ALL4 çıktısını kullan."
)